# 8 . Support / Resistance zones — a look

The **plain 200-MA breakout** strategy from notebook 7 (one rule, nothing else) with the
**S/R zones drawn on top**, so you can see how the zones behave before designing strategies
around them.

### How the zones are drawn (and why it looks like that)
Zones come from **swing pivots clustered by price**. A zone's **touch count** is how many pivots
landed in it. They are drawn as horizontal segments that:

- **start at the bar the zone's first pivot was *confirmed*** — never earlier. A pivot is only
  confirmed `right` bars after it happens, so a zone genuinely did not exist before that bar.
- **step to a new level when a later pivot merges into it** (the level is a running mean), which
  is why a zone's line can nudge up or down over time.
- are **grouped into strength tiers by touch count** — thicker/more opaque = more touches. Each
  tier is its own legend entry, so you can hide the weak ones.

This is deliberately **causal**: it shows what was known at each bar rather than a flattering
end-state snapshot. That's why zones appear mid-chart and drift — a static "here are the levels"
overlay would look cleaner and be lookahead-cheating.

> Touch count is only *one* notion of strength — you already spotted that a level touched once
> that rejects sharply can matter more. Use this view to judge what "major" should actually mean
> before we harden the scoring.

## 0 . Setup

In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import pandas as pd
from quant.data import get_ohlcv
from quant.engine import BacktestConfig
from quant.reporting import print_summary
from quant.indicators import sr_zone_table
from quant.viz import ResearchChart, heikin_ashi_candles, open_in_browser
from quant.strategies import RuleStrategy, Rule, MA, HeikinAshi, Swings
from quant.strategies.conditions import full_above, full_below, ref_col_stop
pd.set_option('display.max_rows', 200)
print('quant', quant.__version__)

## 1 . Data

In [ ]:
SYMBOL, SOURCE, MARKET = 'XAUUSD', 'dukascopy', 'cfd'
START, END = '2026-07-19', '2026-07-25'
df = get_ohlcv(SYMBOL, '1m', start=START, end=END, source=SOURCE, market=MARKET, tz='UTC')
print(f'{len(df):,} 1m bars  {df.open_time.min()} -> {df.open_time.max()}')

## 2 . The plain 200-MA strategy (one rule)
Enter when a candle forms fully **above** the 200-MA; exit when one forms fully **below** it.
One entry per crossing, stop at the previous swing low. Nothing about S/R feeds the rules yet —
the zones are purely a visual layer for now.

In [ ]:
ha = HeikinAshi('ha')
sw = Swings('sw', 5, 5)
trend = MA('trend', 200, type='sma', smooth=9, smooth_type='ema')   # the 200 MA
trend20 = MA('trend20', 20, type='sma', smooth=9, smooth_type='ema')
trend50 = MA('trend50', 50, type='sma', smooth=9, smooth_type='ema')
trend100 = MA('trend100', 100, type='sma', smooth=9, smooth_type='ema')
trend150 = MA('trend150', 150, type='sma', smooth=9, smooth_type='ema')

ENTRY_SRC, ENTRY_N = 'real', 4      # 'real' = normal candles | ha = Heikin Ashi
EXIT_SRC,  EXIT_N  = 'real', 8

strat = RuleStrategy(
    indicators=[ha, sw, trend, trend20, trend50, trend100, trend150],
    reset=full_below(trend, EXIT_N, src=EXIT_SRC),
    rules=[Rule('ma200_breakout',
                entry=full_above(trend, ENTRY_N, src=ENTRY_SRC),
                exit=full_below(trend, EXIT_N, src=EXIT_SRC),
                stop=ref_col_stop(sw.last_low, fallback_pct=1.0),
                once=True)],
)
print(strat.describe())

In [ ]:
cfg = BacktestConfig(
    initial_cash=10_000, spread=0.12, slippage_bps=1.0, exit_enabled=True,
    sl_mode='none', tp_mode='none',          # the stop comes from the Rule
    sizing_mode='risk_pct_equity', sizing_value=1.0, allow_rule_close=True,
)
res = strat.backtest(df, cfg)
print_summary(res, df=df)

## 3 . Build the S/R zones
`LEFT/RIGHT` = how local a pivot must be (bigger = fewer, more significant swings).
`CLUSTER_PCT` = how close two pivots must be (in %) to count as the *same* zone.

In [ ]:
LEFT, RIGHT, CLUSTER_PCT = 10, 10, 0.15

zones = sr_zone_table(df, LEFT, RIGHT, cluster_pct=CLUSTER_PCT)
print(f'{zones.zone.nunique()} distinct zones, {len(zones)} level-segments '
      f'(a zone gets a new segment each time a pivot merges into it)')

# strongest zones by touch count, with the bar range each was alive
summary = (zones.groupby('zone')
                 .agg(price=('price', 'last'), touches=('touches', 'max'),
                      first_seen=('start_t', 'min'), last_seen=('end_t', 'max'))
                 .sort_values('touches', ascending=False))
summary.round({'price': 2})

## 4 . Chart — price + 200 MA + trades + S/R zones
Purple horizontals are the zones; **thicker/darker = more touches**. Hover any zone for its level
and touch count. Toggle tiers (or the trades) off in the legend to declutter, and switch
Real/Heikin-Ashi candles with the buttons top-right.

In [ ]:
prep = strat.prepare(df)
ch = ResearchChart(prep, candles=True, initial_bars=len(prep), candle_target=len(prep),
                   title=f'{SYMBOL} 1m - 200MA breakout + S/R zones', candle_name='Real candles')
ch.add_indicator(trend)
ch.add_indicator(trend20)
ch.add_indicator(trend50)
ch.add_indicator(trend100)
ch.add_indicator(trend150)
ch.add_sr_zones(zones)                      # causal zone segments, tiered by touch count
ch.add_candle_alt(heikin_ashi_candles(df), name='Heikin Ashi')
ch.add_trades(res.trades)
print('chart ->', open_in_browser(ch.show(), 'sr_zones_chart.html'))

## 5 . What to look for
While reading the chart, these are the questions that decide how we score zones:

- Do prices **reverse** at high-touch zones, or do those zones tend to get **broken** (the
  absorption effect you flagged)?
- Are the sharpest bounces at zones with **few** touches?
- Is `CLUSTER_PCT` sane for gold at this timeframe — are separate levels being merged, or is one
  real level split into several?
- Do bigger `LEFT/RIGHT` produce fewer but more meaningful zones?

Tune the knobs in section 3 and re-run to compare. Once you know what "major" should mean, we can
add reaction-strength scoring and turn the zones into entry/exit conditions
(`near_support`, `bounce_off_support`, `SRZones(...)` as a tagged indicator).